# 03. 100m 격자별 비물리적지수 결합

$$C_i=C_{d(i)},\qquad N_i=0.5D_i+0.5C_i$$
기존 격자 중심점 기준 행정동을 그대로 사용하며 통신점수를 인구비중으로 배분하지 않는다.
인구 0인 격자에도 식의 값은 계산하되 `대상인구존재=False`로 후보에서 제외한다.
최소 인구 격자의 규모점수가 0이어도 인구가 있으면 후보에 포함한다.

## 행정동 연결과 경계 가정
- 개포3동 → 통신 일원2동: 명칭 변경.
- 상일1동 → 통신 상일동: 명칭 변경.
- 상일2동 → 통신 강일동: 분동 전 관할 동 점수의 **대리 적용**.
- 항동 → 통신 오류2동: 분동 전 관할 동 점수의 **대리 적용**.
- 강일동·오류2동도 명칭은 같지만 통신 원본이 분동 전 경계를 나타낼 가능성이 있어 함께 대리 적용으로 표시한다.

공식 연혁은 관할 변화의 근거이지, 2025년 통신파일의 정확한 집계 경계를 보증하지 않는다.
따라서 대리 적용 포함 전체 지수와, 이 네 동의 지수를 결측으로 남긴 보수적 검토용 지수를 함께 저장한다.
제공기관 경계 확인 전 전체 결과는 탐색적 결과다. 나머지 동일명 동도 정밀한 경계 대조를 완료했다는 뜻은 아니다.

근거:
- [개포3동 명칭 변경](https://www.gangnam.go.kr/board/article/7193/view.do?mid=ID01_05013)
- [상일1동 연혁](https://www.gangdong.go.kr/web/dongrenew/contents/sangil_010_010)
- [상일2동은 강일동에서 분동](https://www.gangdong.go.kr/web/newportal/press/10794)
- [항동은 오류2동에서 분동](https://www.guro.go.kr/www/contents.do?key=2079)

## 후속 물리적 분석
100m 격자별 출발점 → 검증된 시설 출입구. 접근성이 나쁠수록 $P_i$가 커지도록 통일한다.
후보 $\mathcal S=\{i:T_i>0,P_i\geq\tau\}$ 안에서 $F_i=0.5P_i+0.5N_i$를 계산한다.
$\tau$는 미정이며 이 노트북의 순위는 비물리적 순위일 뿐 최종 선정지역이 아니다.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

start = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in [start, *start.parents]
                    if (p / 'data/raw').is_dir() and (p / 'code').is_dir())
OUT = PROJECT_ROOT / 'data/processed/nonphysical_index'
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 160)

def rank01(values):
    # 결측을 정상 점수로 바꾸지 않는다. 비교대상이 하나면 순위를 정의할 수 없다.
    assert values.notna().all() and np.isfinite(values).all()
    assert len(values) > 1, '순위 비교대상이 2개 미만입니다.'
    return (values.rank(method='average') - 1) / (len(values) - 1)

def save_checked(frame, name, keys):
    path = OUT / name
    assert not frame.duplicated(keys).any()
    frame.to_csv(path, index=False, encoding='utf-8-sig')
    saved = pd.read_csv(path, encoding='utf-8-sig')
    assert len(saved) == len(frame) and not saved.duplicated(keys).any()
    for col in frame.select_dtypes(include='number'):
        assert np.allclose(frame[col], saved[col], equal_nan=True), col
    print('저장 및 재검증:', path.relative_to(PROJECT_ROOT))
    return saved

## 1. 앞 단계 결과와 명시적 연결표

In [2]:
grid = pd.read_csv(OUT / 'grid_senior_population_score.csv', encoding='utf-8-sig')
dong = pd.read_csv(OUT / 'dong_communication_score.csv', encoding='utf-8-sig', dtype={'행정동코드':str})
assert grid.notna().all().all() and dong.notna().all().all()
crosswalk = grid[['시군구','행정동']].drop_duplicates().copy()
crosswalk['통신행정동'] = crosswalk['행정동']
crosswalk['연결방식'] = '동일명 연결'
crosswalk['경계대리적용'] = False
overrides = {
    ('강남구','개포3동'): ('일원2동','명칭 변경',False),
    ('강동구','상일1동'): ('상일동','명칭 변경',False),
    ('강동구','상일2동'): ('강일동','분동 전 관할 동 대리 적용',True),
    ('구로구','항동'): ('오류2동','분동 전 관할 동 대리 적용',True),
    ('강동구','강일동'): ('강일동','분동 전 경계 가능성',True),
    ('구로구','오류2동'): ('오류2동','분동 전 경계 가능성',True),
}
for (gu, name), (old, method, proxy) in overrides.items():
    mask = crosswalk['시군구'].eq(gu) & crosswalk['행정동'].eq(name)
    assert mask.sum() == 1
    crosswalk.loc[mask, ['통신행정동','연결방식','경계대리적용']] = [old,method,proxy]
lookup = dong.rename(columns={'자치구':'시군구','행정동':'통신행정동','행정동코드':'통신행정동코드'})
crosswalk = crosswalk.merge(lookup, on=['시군구','통신행정동'], how='left', validate='many_to_one')
assert crosswalk['커뮤니케이션부족도'].notna().all(), '알려진 대응 외 미연결 동 확인 필요'
display(crosswalk.loc[crosswalk['연결방식'].ne('동일명 연결')])

,시군구,행정동,통신행정동,연결방식,경계대리적용,통신행정동코드,커뮤니케이션부족도,통신60세이상인구
6,구로구,항동,오류2동,분동 전 관할 동 대리 적용,True,1117068,0.512188,12816.0
10,구로구,오류2동,오류2동,분동 전 경계 가능성,True,1117068,0.512188,12816.0
347,강남구,개포3동,일원2동,명칭 변경,False,1123074,0.433095,4549.0
423,강동구,상일1동,상일동,명칭 변경,False,1125052,0.425456,7762.0
424,강동구,강일동,강일동,분동 전 경계 가능성,True,1125051,0.458789,11164.0
425,강동구,상일2동,강일동,분동 전 관할 동 대리 적용,True,1125051,0.458789,11164.0


## 2. 격자별 점수 및 공간 연결 키

In [3]:
result = grid.merge(crosswalk, on=['시군구','행정동'], how='left', validate='many_to_one')
assert result.notna().all().all()
result['비물리적지수'] = .5 * result['인구규모점수'] + .5 * result['커뮤니케이션부족도']
result['비물리적지수_대리적용제외'] = result['비물리적지수'].mask(result['경계대리적용'])
result['비물리적순위'] = np.nan
eligible = result['대상인구존재'].eq(True)
result.loc[eligible, '비물리적순위'] = result.loc[eligible, '비물리적지수'].rank(ascending=False, method='average')

# 원본 GPKG는 읽기 전용. 좌표는 위경도가 아니라 원본 투영좌표계의 중심점이다.
import sqlite3
spatial_path = PROJECT_ROOT / 'data/processed/spatial/서울시_격자_100m_행정동_기본테이블.gpkg'
with sqlite3.connect(spatial_path.as_uri() + '?mode=ro', uri=True) as connection:
    table, srs_id = connection.execute("SELECT table_name,srs_id FROM gpkg_contents WHERE data_type='features'").fetchone()
    quoted = '"' + table.replace('"','""') + '"'
    attrs = pd.read_sql_query('SELECT GRID_CD, 행정동코드, 시군구, 행정동, 중심점_x, 중심점_y FROM ' + quoted, connection)
    crs = connection.execute('SELECT organization,organization_coordsys_id,definition FROM gpkg_spatial_ref_sys WHERE srs_id=?', (srs_id,)).fetchone()
assert set(attrs['GRID_CD']) == set(result['GRID_CD'])
spatial_check = result[['GRID_CD','시군구','행정동']].merge(attrs, on='GRID_CD', validate='one_to_one', suffixes=('_인구','_공간'))
assert spatial_check['행정동_인구'].eq(spatial_check['행정동_공간']).all()
assert spatial_check['시군구_인구'].eq(spatial_check['시군구_공간']).all()
result = result.merge(attrs[['GRID_CD','행정동코드','중심점_x','중심점_y']], on='GRID_CD', validate='one_to_one')
assert result[['중심점_x','중심점_y']].notna().all().all()
print('GPKG SRS ID:', srs_id, 'CRS:', crs)

GPKG SRS ID: 100000 CRS: ('NONE', 100000, 'PROJCS["Korea_2000_Korea_Unified_Coordinate_System",GEOGCS["GCS_Korea_2000",DATUM["Korean_Geodetic_Datum_2002",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6737"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",38],PARAMETER["central_meridian",127.5],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",1000000],PARAMETER["false_northing",2000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]')


## 3. 결과·대리 적용 영향·민감도 검증

In [4]:
assert len(result) == len(grid) and not result['GRID_CD'].duplicated().any()
assert result['취약노인수'].sum() == grid['취약노인수'].sum()
assert result[['인구규모점수','커뮤니케이션부족도','비물리적지수']].apply(lambda x:x.between(0,1).all()).all()
assert result.groupby(['시군구','행정동'])['커뮤니케이션부족도'].nunique().eq(1).all()
assert result['비물리적지수_대리적용제외'].isna().equals(result['경계대리적용'])
assert result['비물리적순위'].isna().equals(~result['대상인구존재'])
display(result.groupby('경계대리적용').agg(격자수=('GRID_CD','size'), 대상격자=('대상인구존재','sum'), 대상인구=('취약노인수','sum')))
display(result.loc[result['대상인구존재'], ['인구규모점수','커뮤니케이션부족도','비물리적지수']].describe())
display(result.loc[result['대상인구존재']].nlargest(10, '비물리적지수')[['GRID_CD','시군구','행정동','취약노인수','인구규모점수','커뮤니케이션부족도','비물리적지수','경계대리적용']])
# 가중치는 0.5/0.5로 고정. 아래 비교는 선택의 안정성 확인용이며 결과를 바꾸지 않는다.
pool = result.loc[result['대상인구존재']].copy()
base = set(pool.loc[pool['비물리적지수'] >= pool['비물리적지수'].quantile(.9), 'GRID_CD'])
rows = []
for alpha in [.3,.5,.7]:
    alternative = alpha * pool['인구규모점수'] + (1-alpha) * pool['커뮤니케이션부족도']
    selected = set(pool.loc[alternative >= alternative.quantile(.9), 'GRID_CD'])
    rows.append({'인구가중치':alpha, '상위10%_Jaccard':len(base & selected)/len(base | selected),
                 '순위상관':pool['비물리적지수'].rank().corr(alternative.rank())})
display(pd.DataFrame(rows))

,격자수,대상격자,대상인구
경계대리적용,,,
False,59730,20971,166473
True,798,292,2275


,인구규모점수,커뮤니케이션부족도,비물리적지수
count,21263.000000,21263.000000,21263.000000
mean,0.500000,0.514329,0.507165
std,0.287806,0.130436,0.180036
min,0.055663,0.085579,0.070621
25%,0.276526,0.417258,0.356889
50%,0.523751,0.528676,0.507764
75%,0.734997,0.608007,0.655929
max,1.000000,0.789558,0.893086


,GRID_CD,시군구,행정동,취약노인수,인구규모점수,커뮤니케이션부족도,비물리적지수,경계대리적용
38497,다사590593,강북구,번2동,66,0.996614,0.789558,0.893086,False
38778,다사591593,강북구,번2동,48,0.993745,0.789558,0.891652,False
37952,다사588586,강북구,번2동,42,0.991652,0.789558,0.890605,False
38217,다사589586,강북구,번2동,42,0.991652,0.789558,0.890605,False
38218,다사589587,강북구,번2동,41,0.991181,0.789558,0.890370,False
38491,다사590587,강북구,번2동,40,0.990641,0.789558,0.890100,False
38495,다사590591,강북구,번2동,40,0.990641,0.789558,0.890100,False
37448,다사586594,강북구,번2동,39,0.990147,0.789558,0.889853,False
38225,다사589594,강북구,번2동,39,0.990147,0.789558,0.889853,False
37703,다사587595,강북구,번2동,37,0.988924,0.789558,0.889241,False


,인구가중치,상위10%_Jaccard,순위상관
0,0.3,0.743957,0.966461
1,0.5,1.000000,1.000000
2,0.7,0.769327,0.985557


## 4. 결과 저장 및 재확인

In [5]:
save_checked(crosswalk, 'dong_communication_crosswalk.csv', ['시군구','행정동'])
saved = save_checked(result, '서울시_100m_비물리적지수.csv', ['GRID_CD'])
assert saved['비물리적지수'].notna().all()
assert saved['취약노인수'].sum() == grid['취약노인수'].sum()
assert saved['비물리적지수_대리적용제외'].isna().sum() == result['경계대리적용'].sum()
print('비물리적지수 계산 완료. 대리 적용 지역은 탐색적 결과이며 물리적·최종 선정지수는 아직 계산하지 않았습니다.')

저장 및 재검증: data\processed\nonphysical_index\dong_communication_crosswalk.csv
저장 및 재검증: data\processed\nonphysical_index\서울시_100m_비물리적지수.csv
비물리적지수 계산 완료. 대리 적용 지역은 탐색적 결과이며 물리적·최종 선정지수는 아직 계산하지 않았습니다.
